# 1. 뉴스 가져오기

## 1.1. 네이트 뉴스 페이지에서 기사 가져오기

In [ ]:
from wrapper.news_fetcher import NewsFetcher
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *


news = NewsFetcher()

news_list = news.fetch_news(n_pages=10)
news.save_csv()

news_tags = [news.tags[i][0] for i in news.tags.keys()]
news_ids = {tag: [] for tag in news_tags}

for tag in news_tags:
    news.news[tag].pop(0)

news = news.news


## 1.2. failback: csv 파일로부터 뉴스 데이터 가져오기

In [ ]:
from wrapper.news_fetcher import NewsFetcher


news = NewsFetcher().load_csv()

# 2. 뉴스 업로드

## 2.1. 업로드

In [ ]:
from wrapper.api_wrapper import ApiWrapper


api = ApiWrapper()
uploaded_news = api.upload_news(news)

## 2.2. failback: 뉴스 ID 복원

In [ ]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
on_server = api.download_news()
uploaded_news: list[UploadedNews] = []

for tag in news:
    for n in news[tag]:
        for o in on_server:
            if o["title"] == n.title:
                uploaded_news.append(
                    UploadedNews(
                        title=n.title,
                        content=n.content,
                        image=n.image,
                        press=n.press,
                        pub_time=n.pub_time,
                        tag=n.tag,
                        url=n.url,
                        id=o["newsIdx"],
                    )
                )
                break

len(uploaded_news)

api.save_csv(uploaded_news)

## 2.3. csv 파일로부터 업로드된 뉴스 불러오기

In [ ]:
from wrapper.api_wrapper import ApiWrapper
from entity.entity import *


api = ApiWrapper()
uploaded_news = api.load_csv()

# 3. 뉴스 요약

## 3.1. 뉴스 요약 진행

In [ ]:
from wrapper.llm_wrapper import LLM
from wrapper.api_wrapper import ApiWrapper
from tqdm import tqdm

from entity.entity import *

import pickle


api = ApiWrapper()


llm = LLM(n_ctx=8192, max_tokens=1024)
summerized_news: list[SummerizedNews] = []

for news in tqdm(uploaded_news):
    llm.set_prompt(
        f"""
        [요청 사항]
        - 이 뉴스를 다음 양식을 준수하는 세 문장으로 요약해 주세요.

        [준수 사항]
        - 첫 번째 문장은 이 기사에서 다루는 핵심 사건을 설명하는 120자 내외의 완결된 문장이어야 합니다.
        - 두 번째 문장은 사건의 배경과 관련된 맥락을 설명하는 120자 내외의 완결된 문장이어야 합니다.
        - 세 번째 문장은 사건의 진행과 결과를 설명하는 120자 내외의 완결된 문장이어야 합니다.

        [참고 사항]
        - 요약하신 자료는 텍스트 임베딩을 거쳐 클러스터링 작업에 사용될 것입니다.
        - 이 뉴스는 {news.tag} 분야의 뉴스입니다.
        - 이 뉴스는 {news.pub_time} 시점에 게시되었습니다.

        [예시]
        1. 지난 23일 16시 경 광주 광산구 아파트 주차장에서 차량 4대를 들이받고 벤츠를 버린 운전자가 사건 발생 12시간 만에 경찰에 자진 출석했다.
        2. 사고 직후 운전자는 아무런 조치 없이 연락처와 벤츠를 남기고 도주했으며, 사고 12시간 40분 만에 같은 날 오후 6시쯤 경찰에 출석했다.
        3. 경찰은 A씨를 들이받은 차량을 수습하지 않은 채 도주한 혐의를 적용하여 조사하고 있으며, CCTV 등을 통해 운전 경로를 추적 수사할 방침이다.

        """
    )

    content = llm.generate(
        instruction=
        f"""
        [뉴스 제목]
        {news.title}
        [뉴스 내용]
        {news.content}
        """[:8191],
        reset_prompt=True
    )

    summerized_news.append(SummerizedNews(title=news.title, content=content, topics="", id=news.id))


with open("summerized.pkl", "wb") as f:
    pickle.dump(summerized_news, f)


def get_summerized_news(id: int) -> SummerizedNews:
    for i in summerized_news:
        if i.id == id:
            return i
    
    return None


for i in summerized_news[:5]:
    print(i.content, end="\n\n")

## 3.3. failback: 요약된 뉴스 불러오기

In [ ]:
import pickle
from entity.entity import *

with open("summerized.pkl", "rb") as f:
    summerized_news = pickle.load(f)


def get_summerized_news(id: int) -> SummerizedNews:
    for i in summerized_news:
        if i.id == id:
            return i
    
    return None

# 4. 임베딩

## 4.1. 요약문 임베딩

In [ ]:
from wrapper.llm_wrapper import LLM
import numpy as np
from tqdm import tqdm

clusters = {}
llm = LLM(embedding=True)

firsts = []
seconds = []
thirds = []

for texts in tqdm(summerized_news):
    sentences = list(filter(lambda x: x.strip() != '', texts.content.split('\n')))
    first, second, third = llm.embed(sentences)
    
    firsts.append(np.mean(first, axis=0))
    seconds.append(np.mean(second, axis=0))
    thirds.append(np.mean(third, axis=0))

assert len(firsts) == len(seconds) == len(thirds)

# first_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in firsts) - len(embedding)), 'constant') for embedding in firsts])
# second_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in seconds) - len(embedding)), 'constant') for embedding in seconds])
# third_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in thirds) - len(embedding)), 'constant') for embedding in thirds])

first_embeddings = np.array(firsts)
second_embeddings = np.array(seconds)
third_embeddings = np.array(thirds)


# 5. 유사도

## 5.1. cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity


# 각 문장별 유사도 행렬 계산
first_similarity = 1 - cosine_similarity(first_embeddings)
second_similarity = 1 - cosine_similarity(second_embeddings)
third_similarity = 1 - cosine_similarity(third_embeddings)

# 세 유사도의 평균값을 최종 유사도로 사용
similarity_matrix = (first_similarity + second_similarity + third_similarity) / 3
metric="precomputed"

## 5.2. pairwise distance(euclidian)

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import Normalizer


norm = Normalizer()
first_embeddings = norm.fit_transform(first_embeddings)
second_embeddings = norm.fit_transform(second_embeddings)
third_embeddings = norm.fit_transform(third_embeddings)

first_similarity = pairwise_distances(first_embeddings)
second_similarity = pairwise_distances(second_embeddings)
third_similarity = pairwise_distances(third_embeddings)

# 세 유사도의 평균값을 최종 유사도로 사용
similarity_matrix = (first_similarity + second_similarity + third_similarity) / 3
metric="euclidean"

# 6. 클러스터링

## 6.1. DBSCAN

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity


dbscan = DBSCAN(eps=0.9, min_samples=4, metric="precomputed")
clusters_ = dbscan.fit_predict(similarity_matrix)

# 클러스터 결과
print(len(set(clusters_)))
clusters = {}  # 클러스터를 저장할 딕셔너리

for cluster_id in set(clusters_):
    cluster_news_ids = set()  # 중복을 피하기 위해 set 사용

    if cluster_id != -1:  # -1은 노이즈
        print(f"Cluster {cluster_id}:")
        for i in np.where(clusters_ == cluster_id)[0]:
            # 중복된 ID를 추가하지 않도록 set에 추가
            cluster_news_ids.add(summerized_news[i].id)
            print(summerized_news[i].id, summerized_news[i].title)

        clusters[cluster_id] = list(cluster_news_ids)


# ?. legacy

In [ ]:
# from wrapper.llm_wrapper import LLM
# model = LLM(embedding=True).model
from llama_cpp import Llama

MODEL_PATH = "/home/gpp/src/model/llama3-korean-bllossom-8b/llama-3-Korean-Bllossom-8B-Q4_K_M.gguf"
model = Llama(model_path=MODEL_PATH, embedding=True, verbose=False)
clusters = {}

In [ ]:
import numpy as np
from tqdm import tqdm

firsts = [  ]
seconds = [ ]
thirds = [  ]

for texts in tqdm(summerized_news):
    first, second, third = list(filter(lambda x: x.strip() != '', texts.content.split('\n')))
    firsts.append(np.mean(model.embed(first, normalize=True), axis=0))
    seconds.append(np.mean(model.embed(second, normalize=True), axis=0))
    thirds.append(np.mean(model.embed(third, normalize=True), axis=0))

assert len(firsts) == len(seconds) == len(thirds)

first_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in firsts) - len(embedding)), 'constant') for embedding in firsts])
second_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in seconds) - len(embedding)), 'constant') for embedding in seconds])
third_embeddings = np.array([np.pad(embedding, (0, max(len(e) for e in thirds) - len(embedding)), 'constant') for embedding in thirds])

In [ ]:
from sklearn.metrics import pairwise_distances
from sklearn.preprocessing import Normalizer


norm = Normalizer()
first_embeddings = norm.fit_transform(first_embeddings)
second_embeddings = norm.fit_transform(second_embeddings)
third_embeddings = norm.fit_transform(third_embeddings)

first_similarity = pairwise_distances(first_embeddings)
second_similarity = pairwise_distances(second_embeddings)
third_similarity = pairwise_distances(third_embeddings)

# 세 유사도의 평균값을 최종 유사도로 사용
similarity_matrix = (first_similarity + second_similarity + third_similarity) / 3
metric="euclidean"

In [ ]:
from sklearn.cluster import DBSCAN
from sklearn.metrics.pairwise import cosine_similarity


dbscan = DBSCAN(eps=0.4, min_samples=3, metric="precomputed")
clusters_ = dbscan.fit_predict(similarity_matrix)

# 클러스터 결과
# print(len(set(clusters_)))
clusters = {}  # 클러스터를 저장할 딕셔너리

for cluster_id in set(clusters_):
    cluster_news_ids = set()  # 중복을 피하기 위해 set 사용

    if cluster_id != -1:  # -1은 노이즈
        print(f"{cluster_id}")
        for i in np.where(clusters_ == cluster_id)[0]:
            # 중복된 ID를 추가하지 않도록 set에 추가
            cluster_news_ids.add(summerized_news[i].id)
            print('\t' + summerized_news[i].title)

        clusters[cluster_id] = list(cluster_news_ids)


In [ ]:
llm.set_prompt(
    f"""
    [요청 사항]
    다음 클러스터 중에 가장 잘 분류된 클러스터의 ID를 출력하십시오.

    [제약 사항]
    클러스터의 ID만을 출력하십시오.
    """
)

llm.generate(
    """
0
	중남미 순방길 오르는 윤 대통령, 2년 만에 한·중 정상회담 가능성
	"트럼프 2기도 한미동맹 굳건"…'물밑 외교전' 시작됐다 [尹대통령 남미 순방]
	14일부터 남미 순방…트럼프와 회동 추진
	윤 대통령, APEC·G20 참석…한중·한미일 회담 추진
	尹, 시진핑·이시바 이어 트럼프 만날까
1
	김정은도 '북·러 조약' 서명…북한군 전장 투입 공식화만 남았다
	김정은, 북러 조약 비준…"파병 규모 늘지 않을 것"
	푸틴 이어 김정은도 '북-러 조약' 비준…'파병 공식화'로 이어지나
	북한도 '전쟁지원' 북러조약 비준…파병 공식화 가능성
	김정은도 북·러조약 서명 우크라戰 본격 투입 임박
	푸틴도 김정은도 '군사동맹' 북러조약 서명…'파병 공식화' 임박
2
	'원전 예산' 원안 통과에…한동훈 "민주당, 탈원전 잘못 인정"
	한동훈, 원전 예산 상임위 합의에 "野, 탈원전 정책 한계 인정"
	한동훈, 원전 예산 정부안 통과에 "민주당도 탈원전 잘못 인정"
	'개식용 종식' 예산…여 "이재명도 공약" 야 "살처분이 보호냐"
	한동훈, 원전예산 상임위 합의에 "민주, 탈원전 잘못 인정"
	한동훈 "드디어 민주당도 탈원전이 잘못된 걸 인정"
	'개식용 종식' 예산에 與 "이재명도 공약" 野 "살처분 정책"
3
	정연욱, 이기흥 3선 출마 승인한 스포츠공정위 불공정성 강력 비판
	정연욱 의원 "이기흥 체육회장 연임도전 승인? 이게 공정인가?"
	'이기흥 3선 연임' 길 터준 대한체육회에 뿔난 정부…"심히 유감"
	정연욱 "스포츠공정위의 이기흥 대한체육회장 3선 승인은 짜고 치는 심사"
4
	김종인 "대통령 되는 순간, 친구·가족 개념 떠나야" 연이틀 쓴소리
	한동훈, 윤 '어쨌든 사과' 뒤 이재명 때리기로 급선회…민주 "뻔뻔"
	與 "이재명 재판 생중계하라" 막판 공세
	여, 긴급 대책 회의…이재명 1심 총공세
	야, 이재명 수호 총력전…'특검 수정안' 압박
	집회·탄원 독려…이재명 1심 선고 앞 친명계 전방위 사법부 압박 '눈총'
	조응천 "윤, 부부싸움 아니고 이혼할 결심해야"
	이재명 선고 임박…여 "수험생 짜증" 야 "무죄"
	'이재명 무죄 판결' 압박 동원정치가 보여준 '일극체제' 민주당 현실
	"언론에 툭 던지고 갈등 부추기고…" 한동훈 또 저격한 洪
	'이재명 무죄 여론전' 나서는 민주당 속내는…與는 '생중계'로 맞불
	한동훈 "대입 논술 날까지 시위"…재판 생중계 촉구
	대법원장 추천 '비토권'…민주당, 채 해병 국정조사 병행
	민주 "윤 대통령, 골프 친 것 들통 나니 외교로 포장"
	이재명 1심 결과·명태균 구속 여부, 與野 정치적 명운 가른다
5
	'국방부 장관 정책 보좌관' 출신이 본 트럼프 "방위비 협상 당시 정말 미치는 줄"
	부승찬 "與 이탈표? 회의적…尹-韓 운명공동체, 같이 무너질 것"
	친윤 강명구 "용산 참모진, 尹 따라 일정 책임 필요…金여사 특검법 통과? 탄핵의 시작"
	김용태 "한동훈 5대 요구, 매듭 중" 김한규 "특검 동요 없다? 한동훈 측 항복"
	'한동훈 동명이인 댓글 의혹' 대응에 잠잠한 與, 왜?[이정주의 질문하는기자]
	묻고 더블로 가는 민주당?  김여사 특검 수정안 내고 대법원 예산 240억 증액
6
	검찰, 명태균 "김건희 여사로부터 돈 받아" 진술 확보
	명태균 측 "대화 발단은 이준석"…통화 전 메시지 확보
	[단독] 명태균-이준석 대화 복원…윤리위 대책 '김여사' 언급
	"김영선 좀 해줘라" 말한 '2022년 5월 9일' 무슨 일이?…상황 재구성
	회견서 "축하 전화"랬는데…명태균, 검찰에 "공천 여부 물은 것"
    """
)

# 7. 짜집기 뉴스 제작

## 7.1. (수작업) 가장 잘 군집화 된 군집 선택

In [ ]:
top_clusters = [clusters[0]] #, clusters[2], clusters[6]]

In [ ]:
from wrapper.llm_wrapper import LLM
from entity.entity import Article

# del model
# del llm
llm = LLM(max_tokens=8192)#, temperature=0.6, top_p=0.7)


def news_creation_chain(llm: LLM, news_contents: str) -> str:
    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 요약들을 종합하여 15개의 문장으로 정리해 주세요.

        [준수 사항]
        각 문장은 ~했어요, ~해요로 종결되는 300자 내외의 완결된 문장이어야 합니다.
        핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어야 합니다.


        [참고 사항]
        하나의 뉴스 요약에서, 첫 번째 문장은 이 기사에서 다루는 핵심 사건, 두 번째 문장은 사건의 배경과 관련된 맥락, 세 번째 문장은 사건의 진행과 결과를 설명합니다.
        """
    )
    processed = llm.generate(news_contents)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 뉴스 초고를 완결된 3문단 구성의 뉴스 기사로 만들어 주세요.

        [준수 사항]
        뉴스 기사는 고등학생들이 읽을 것이에요. 친근한 말투 부탁해요.
        뉴스 기사는 3문단으로 구성되어야 해요.
        뉴스 기사의 모든 문장은 친근한 말투인 ~했어요, ~해요로 종결되어야 해요.
        뉴스 기사의 모든 문장은 300자 내외의 완결된 문장이어야 해요.

        [참고 사항]
        이 초고는 핵심 사건에 대한 기사 문장 5문장, 사건의 배경과 관련된 맥락에 대한 문장 5문장, 사건의 진행과 결과를 설명하는 문장 5문장으로 구성되어 있어요.
        """
    )
    processed = llm.generate(processed)

    llm.set_prompt(
        f"""
        [요청 사항]
        다음 문장들에 대하여, 모든 문장을 ~했어요, ~해요, ~어요로 종결되게 수정해 주세요.

        [예시]
        최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있습니다. -> 최근 집콕 트렌드가 확산되면서 홈 인테리어와 관련된 라이프스타일 시장이 급성장하고 있어요.
        """
    )

    result = llm.generate(processed)
    return result

news_contents: list = []
finals: list[Article] = []
for cluster in top_clusters:
    for news_id in cluster:
        target_news = get_summerized_news(news_id)
        print(target_news.title)
        news_contents.append(target_news.content)

    result = news_creation_chain(llm, '\n\n'.join(i[2:] for i in news_contents))
    del llm

    finals.append(Article(title=target_news.title, content=result, news_id=cluster[:]))


In [ ]:
finals

In [ ]:
result = api.upload_article(finals[0])
print(result.status_code)

In [ ]:
articleIdx = result.json()["data"][0]["articleIdx"]

In [ ]:
article = finals[0]

In [ ]:
ones, twos, threes = [], [], []

for i in article.news_id:
    one, two, three = get_summerized_news(i).content.split('\n')
    ones.append(one.split('.')[1].strip())
    twos.append(two.split('.')[1].strip())
    threes.append(three.split('.')[1].strip())

In [ ]:
llm = LLM(max_tokens=2048, temperature=0.7, top_p=0.7)

llm.set_prompt(
    f"""
    [요청 사항]
    다음 내용으로부터 80자 이내의 완결된 문장을 하나 만들어 주세요.
    문장을 완결된 문장이어야 하고, 작은 따옴표로 강조 표시된 부분이 포함되어야 합니다.

    [제약 사항]
    핵심 키워드 또는 중심 음절은 작은 따옴표로 강조 표시하여야 합니다.
    문장은 완결체로 종결되어야 합니다.

    [예시]
    미국과 일본이 체결한 '안보 협력 강화 협정'이 공식 발효됨에 따라 양국 군대의 인도-태평양 지역 합동 훈련이 본격화될 전망이다.
    """
)

quiz = llm.generate(
    '\n'.join(ones)
    + '\n'.join(twos)
    + '\n'.join(threes)
)

quiz

In [ ]:
llm = LLM(max_tokens=2048, temperature=0.9, top_p=0.5)

llm.set_prompt(
    f"""
    [요청 사항]
    이 문장들으로부터 사실이 아닌 80자 분량의 문장을 하나 지어 주세요.
    """
)

llm.generate(
    '\n'.join(ones)
    + '\n'.join(twos)
    + '\n'.join(threes)
)

In [ ]:
tmp = quiz[quiz.find("'") + 1:]
keyword = tmp[:tmp.find("'")]
keyword

In [ ]:
quiz_result = quiz.replace("'" + keyword + "'", "_" * len(keyword))
quiz_result

In [ ]:
llm = LLM(max_tokens=2048, temperature=0.9, top_p=0.6)


partial_knowledge = ones
llm.set_prompt(
    f"""
    [요청 사항]
    다음 문장에서 빈 칸에 들어갈 답만을 말씀해 주세요.

    [제약 사항]
    정답의 글자 수는 {quiz_result.count('_')} 내외여야 합니다.
    세 가지의 가능한 답만을 제시하여야 합니다.
    각 답은 완전히 다른 아이디어나 관점을 제공해야 합니다
    
    [배경 지식]
    {partial_knowledge}
    """
)

blank = llm.generate(
    quiz_result
)

blank

In [ ]:
answer_contents = [i[i.find('. ') + 1:].strip() for i in blank.split('\n')]
answer_contents.append(keyword)
answer_contents